# Finetuning Llama Models

## Model
unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit

## Dataset

In [1]:
!nvidia-smi

Wed May 13 14:54:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.12              Driver Version: 550.90.12      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:00:06.0 Off |                    0 |
| 30%   31C    P8             27W /  300W |       2MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import warnings
import logging

# Silence standard warnings
warnings.filterwarnings("ignore")

os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["UNSLOTH_FORCE_RICH"] = "1"

# Silence Hugging Face Transformers logging
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_warning()

# Silence standard logging (useful for multiproc logs)
logging.getLogger("abc").setLevel(logging.ERROR)

## Fixed Code Blocks

- Load and prepare dataset
- Save fine tuned model inferences to disk
- Use saved inferences file for evaluation via LLM Judge

### Dataset Preparation

In [5]:
from datasets import load_dataset

print("loading the dataset...")

# Load the dataset
dataset = load_dataset(
    "json",
    data_files="/workspace/data/llm_judge_results.jsonl", 
    split="train"
)

# Filter the dataset to take very high quality records
filtered_dataset = dataset.filter(
    lambda x: x["judge_evaluation"]["overall_score"] > 6 and not
    x["judge_evaluation"]["hallucination_detected"]
)

print(f"len(dataset) - {len(dataset)} records")
print(f"len(filtered_datset) - {len(filtered_dataset)} records")

shuffled_dataset = filtered_dataset.shuffle(seed=42)
split_dataset = shuffled_dataset.train_test_split(test_size=0.1, seed=42)

# Dry run selection - best 500 for training, proportional 50 for eval
train_set = split_dataset["train"]
test_set = split_dataset["test"]

# Save to disk
train_set.save_to_disk("/mnt/datasets/medgemma_extracts/train")
test_set.save_to_disk("/mnt/datasets/medgemma_extracts/test")

print(f"len(train_set) - {len(train_set)} records")
print(f"len(test_set) - {len(test_set)} records")

loading the dataset...


Filter:   0%|          | 0/4327 [00:00<?, ? examples/s]

len(dataset) - 4327 records
len(filtered_datset) - 4324 records


Saving the dataset (0/1 shards):   0%|          | 0/3891 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/433 [00:00<?, ? examples/s]

len(train_set) - 3891 records
len(test_set) - 433 records


### Defining The Exact Schema
```json
{
    "summary": "",
    "clinical_reasoning": "",
    "relationships": [
        {
            "subject": "",
            "predicate": "",
            "object": "",
            "polarity": "",
            "certainty": "",
            "evidence": ""
        }
    ],
    "keywords": [""]
}
```

In [4]:
JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

### Inference And Save To Disk
If required, we can also run a short programmatic evaluation logic to ensure the quality of generation is high before the accuracy of generation can be evaluated by judge

In [7]:
import os
import ast
import json
import torch
from tqdm.auto import tqdm

def evaluate_local(model, tokenizer, dataset, output_file, json_schema=JSON_SCHEMA, num_samples=50):
    """
    Performs inference on a slice of evaluation split
    Evaluate the generations programmatically
    Save results to JSONL file for LLM JUDGE evaluation
    """
    required_keys = {"summary", "clinical_reasoning", "relationships", "keywords"}
    required_rel_keys = {
        "subject",
        "predicate",
        "object",
        "polarity",
        "certainty",
        "evidence"
    }

    syntax_errors = 0
    schema_errors = 0

    # Ensure output file is fresh for each run
    if os.path.exists(output_file):
        print(f"Deleting old output file {output_file} from disk...")
        os.remove(output_file)

    print(f"Starting local evaluation on {num_samples} records...")

    for i in tqdm(range(num_samples), desc="inference progress", leave=True):
        row = dataset[i]
        raw_text = row["raw_medical_text"]
        ground_truth = row["data"]
        record_id = row["id"]

        messages = [
            {
                "role": "system", 
                "content": f"""
                Extract data strictly into this JSON schema\n\n{json_schema}\n\nOutput ONLY raw JSON without preamble.\n
                CRITICAL
                --------
                Use ONLY double quotes (") for JSON keys and string values. Do not use single quotes ('). Ensure internal apostrophes in clinical terms (e.g., patient's) remain as-is.
                """
            },
            {
                "role": "user", 
                "content": f"CONTEXT: {raw_text}"
            },
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True, # added in latest fix
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs, # original - input_ids=inputs
                max_new_tokens=2048,
                use_cache=True,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        prediction = tokenizer.batch_decode(
            outputs[:, inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )[0]
        clean_pred = prediction.strip().replace("```json", "").replace("```", "")

        # --- LOCAL EVALUATION ---
        parsed_obj = None
        current_record_syntax_error = False

        try:
            parsed_obj = json.loads(clean_pred)
        except json.JSONDecodeError:
            try:
                parsed_obj = ast.literal_eval(clean_pred)
            except (ValueError, SyntaxError):
                syntax_errors += 1
                current_record_syntax_error = True

        if not current_record_syntax_error:
                is_schema_invalid = False

                # Check root keys
                if set(parsed_obj.keys()) != required_keys:
                    is_schema_invalid = True
                else:
                    relations = parsed_obj.get("relationships", [])
                    if not isinstance(relations, list) or len(relations) == 0:
                        is_schema_invalid = True
                    else:
                        # Check for relationship triplets
                        for rel in relations:
                            if not isinstance(rel, dict) or set(rel.keys()) != required_rel_keys:
                                is_schema_invalid = True
                                break

                if is_schema_invalid:
                    schema_errors += 1

        # --- SAVE TO FILE ---
        record = {
            "id": record_id,
            "input_context": raw_text,
            "student_output": clean_pred,
            "ground_truth": ground_truth
        }

        with open(output_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
    f.close()

    # --- CALCULATE METRICS ---
    syntax_success_rate = ((num_samples - syntax_errors) / num_samples) * 100
    schema_adherence_rate = ((num_samples - schema_errors) / num_samples) * 100
    
    print(f"\n--- LOCAL EVALUATION COMPLETE ---")
    print(f"Syntax Success Rate: {syntax_success_rate}%")
    print(f"Schema Adherence Rate: {schema_adherence_rate}%")
    print(f"Outputs ready for LLM Judge at: {output_file}")


### Judging With LLM
Use **kosbu/Llama-3.3-70B-Instruct-AWQ** as judge LLM. Alternatively, can also use hosted LLMs from comemrcial providers like **Google**, **OpenAI**, **DeepSeek** etc. 

In [ ]:
import os
import torch
import wandb
import numpy as np
from dotenv import load_dotenv
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template, train_on_responses_only

load_dotenv()

# Using the Dynamic Quantized version for superior JSON logic retention
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True,
    use_gradient_checkpointing=True
)

# Targeting all linear layers to prevent Catastrophic Forgetting
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

# Load llama-3 chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

def format_prompts(examples):
    instructions = examples["raw_medical_text"]
    outputs = examples["data"]
    texts = []

    for instruction, output in zip(instructions, outputs):
        messages = [
            {
                "role": "system",
                "content": """You are a clinical informatician. Extract data strictly into this JSON schema\n\n{JSON_SCHEMA}\n\n Do not include any preamble or markdown code blocks. Output only the raw JSON."""
            },
            {
                "role": "user",
                "content": f"""CONTEXT:\n{instruction}"""
            },
            {
                "role": "assistant",
                "content": output
            }
        ]
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
        )
    return {"text": texts}

train_set_formatted = train_set.map(format_prompts, batched=True)
test_set_formatted = test_set.map(format_prompts, batched=True)

print("\n✅ successfully formatted train and test set for fine tuning")

# Calculate the token length of the train set
token_counts = [len(tokenizer.tokenize(text)) for text in train_set_formatted["text"]]
p95 = np.percentile(token_counts, 95)

print(f"\nℹ️ 95% of your prompts are shorter than {p95} tokens.")

# Initialize WNADB for logging
wandb.init(
    project=os.environ.get("WANDB_PROJECT"),
    name="llama-3.2-3B-Instruct-DR-1",
    tags=["llama3.2", "4bit", "qlora", "dry-run"]
)

print("\n✅ WANDB initialized")

# Initialize and configure SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_set_formatted,
    eval_dataset=test_set_formatted,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=4096,
        dataset_num_proc=2,

        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        optim="adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),

        max_steps=60,

        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,

        logging_steps=4,
        report_to="wandb",
        seed=3407,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,

        output_dir="/workspace/checkpoints/llama_checkpoints",
        per_device_eval_batch_size=1,
        eval_accumulation_steps=8,
        eval_strategy="steps",
        save_steps=8
    )
)
print("\n✅ trainer for fine tuning configured")

# Masking completion - this prevents the model from learning the instructions
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n"
)
print("\n✅ masking complete")

print("\n⏳💡 getting ready to fine tune...\n")

try:
    trainer_stats = trainer.train()
finally:
    wandb.finish()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 44.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✅ successfully formatted train and test set for fine tuning


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: loknezmonzter (loknezmonzter-dev) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
4,0.399208,0.363148
8,0.280515,0.227693
12,0.204871,0.203811
16,0.191838,0.192150
20,0.169088,0.184356
24,0.163796,0.179775
28,0.149718,0.176020
32,0.156767,0.172577
36,0.134719,0.171001
40,0.138478,0.170079


Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-8/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-16/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-32/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-48/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkpoint-56/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/checkpoints/llama_checkpoints/checkp

TrainOutput(global_step=60, training_loss=0.17427554627259573, metrics={'train_runtime': 1482.262, 'train_samples_per_second': 1.295, 'train_steps_per_second': 0.04, 'total_flos': 8.989124607236506e+16, 'train_loss': 0.17427554627259573, 'epoch': 3.768})

In [8]:
from unsloth import FastLanguageModel

# Load model for inference (using checkpoint-60 for best results)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/workspace/checkpoints/llama_checkpoints/checkpoint-60",
    max_seq_length=4096,
    load_in_4bit=True
)
FastLanguageModel.for_inference(model)

# Evaluation of fine tuned llama-3.2-3B-Instruct
evaluate_local(
    model,
    tokenizer,
    test_set,
    output_file="/workspace/outputs/llama-3.2-3B-Instruct-DR.jsonl"
)

==((====))==  Unsloth 2026.5.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 44.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Starting local evaluation on 50 records...


inference progress:   0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t


--- LOCAL EVALUATION COMPLETE ---
Syntax Success Rate: 54.0%
Schema Adherence Rate: 100.0%
Outputs ready for LLM Judge at: /workspace/outputs/llama-3.2-3B-Instruct-DR.jsonl


In [12]:
# RUN THIS TO SEE THE TRUTH
with open("/workspace/outputs/llama-3.2-3B-Instruct-DR.jsonl", "r") as f:
    i = 0
    for line in f:
        data = json.loads(line)
        output = data["student_output"]
        if i < 1:
            try:
                ast.literal_eval(output)
            except Exception as e:
                print(f"--- FAILED RECORD ID: {data['id']} ---")
                print(f"ERROR: {e}")
                print(f"RAW OUTPUT: {output}")
                print("-" * 50)
                i += 1    

--- FAILED RECORD ID: 4623177-1 ---
ERROR: unterminated string literal (detected at line 1) (<unknown>, line 1)
RAW OUTPUT: {'summary': 'A 32-year-old male presented with sudden severe headache, back pain, chest pain, and progressive bilateral weakness, ultimately diagnosed with a whole-spinal epidural hematoma. He underwent emergency surgery, which led to significant recovery and full functional restoration.', 'clinical_reasoning': 'The patient presented with acute neurological deficits (weakness, paralysis, sensory loss) following a traumatic event (sudden headache, back pain, chest pain). Imaging revealed a whole-spinal epidural hematoma, which was surgically evacuated. Postoperative imaging and clinical improvement confirmed the hematoma's role in the neurological deficits, leading to full recovery.','relationships': [{'subject': 'Patient', 'predicate': 'SHOWS_SYMPTOM', 'object':'sudden headache', 'polarity': 'positive', 'certainty': 'confirmed', 'evidence':'sudden headache'}, {'su